# Auditoría de Calidad de Datos y Análisis Exploratorio (EDA) — StreamView Analytics
**Asignatura:** ADY1104 — Visualización de Datos (Duoc UC)  
**Propósito:** Documentar de forma reproducible la integración, limpieza, detección de sesgos estructurales y hallazgos estratégicos de los datasets corporativos `netflix_movies_detailed_up_to_2025.csv` y `netflix_tv_shows_detailed_up_to_2025.csv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

movies_raw = pd.read_csv(BASE_DIR / 'data' / 'netflix_movies_detailed_up_to_2025.csv')
tv_raw = pd.read_csv(BASE_DIR / 'data' / 'netflix_tv_shows_detailed_up_to_2025.csv')
print('Películas:', movies_raw.shape, '| Series:', tv_raw.shape)

In [ ]:
# 1. Verificación de colisiones de show_id y variables degeneradas
overlap = set(movies_raw['show_id']).intersection(set(tv_raw['show_id']))
print(f'IDs repetidos entre Películas y Series: {len(overlap)}')
print('Nulos en duration (Películas):', movies_raw['duration'].isna().sum())
print('Valores únicos en duration (Series):', tv_raw['duration'].unique())
print('Identidad exacta rating == vote_average (Películas):', movies_raw['rating'].equals(movies_raw['vote_average']))
print('Identidad exacta rating == vote_average (Series):', tv_raw['rating'].equals(tv_raw['vote_average']))

In [ ]:
# 2. Limpieza e integración mediante las funciones puras del paquete streamview
from streamview import data, metrics, config as C

df_m = data.clean_frame(movies_raw, C.MOVIE, 'movies.csv')
df_t = data.clean_frame(tv_raw, C.SERIES, 'tv.csv')
full_df = pd.concat([df_m, df_t], ignore_index=True, sort=False)
full_df.insert(0, 'uid', np.arange(len(full_df)))

display(metrics.type_summary(full_df))
for note in metrics.data_quality_notes(full_df):
    print('-', note)

In [ ]:
# 3. Resumen Financiero y Rentabilidad por Género (Películas)
print('KPIs Financieros:', metrics.financial_kpis(full_df))
display(metrics.genre_roi_table(full_df, 10))